# 03 · Statistical forecasts (Theta + AutoETS + SeasonalNaive)

Drives `m5.cv.stats_cv` — the same path `make cv-stats` uses. 
Outputs the rolling-origin CV frame to `artifacts/cv_stats.parquet` so 
`06_score.ipynb` picks it up automatically.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd

from m5.config import SETTINGS, set_global_seed
from m5.cv import stats_cv
from m5.evaluation import compute_components, make_submission, wrmsse_for_models
from m5.models.stats import fit_predict_stats
from m5.plots import configure_style

configure_style()
set_global_seed()

## Cross-validation

In [ ]:
df = pd.read_parquet(SETTINGS.processed_dir / 'long.parquet')

cv = stats_cv(df, h=SETTINGS.horizon, n_windows=SETTINGS.n_windows)
cv.to_parquet(SETTINGS.artifacts_dir / 'cv_stats.parquet', index=False)
cv.tail()

In [ ]:
components = compute_components(df[df['ds'] < cv['ds'].min()])
wrmsse_for_models(cv[['unique_id', 'ds', 'y']], cv, components)

## Train on full data, emit a 28-day forecast

One call per Nixtla — same path as `make forecast-stats`.

In [ ]:
forecast = fit_predict_stats(df, horizon=SETTINGS.horizon)
forecast.to_parquet(SETTINGS.forecasts_dir / 'forecast_stats.parquet', index=False)
forecast.head()

## Optional — Kaggle-format submission per model

Pivot a single model column into the wide submission layout.

In [ ]:
model = 'AutoETS'
submission = make_submission(
    forecast[['unique_id', 'ds', model]], h=SETTINGS.horizon, value_col=model, layout='kaggle',
)
submission.head()